# tarjeeh-ugc-ai — free Kaggle GPU fallback

Runs the **same** open-source pipeline as `hf-space/`, on Kaggle's free GPU (T4 x2 / P100).

**Use this only when HF ZeroGPU is unavailable or its quota is spent.**

> Free resources only. Do not enable any paid Kaggle add-on, and never substitute a paid
> generation API. If the free GPU session ends mid-run, the finished scenes are kept in
> `/kaggle/working/projects/<job_id>/scenes/` and the job resumes from there.

**Before running:** Settings → Accelerator → **GPU T4 x2**, and Internet → **On**.


## 1 · Environment check


In [ ]:
import subprocess, sys, shutil
print('python', sys.version.split()[0])
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip() or 'NO GPU - enable the accelerator')
print('ffmpeg:', shutil.which('ffmpeg'))


## 2 · Dependencies

Open weights and open-source packages only. Nothing here is billable.


In [ ]:
%%capture
!pip install -q --upgrade "diffusers>=0.36.0" "transformers>=4.49.0" "accelerate>=1.1.0" safetensors ftfy
!pip install -q imageio imageio-ffmpeg opencv-python-headless soundfile
!pip install -q kokoro>=0.9.4 openai-whisper
!apt-get -qq install -y ffmpeg > /dev/null


## 3 · Project code

Clone your repo so the notebook uses the committed backend rather than a copy that drifts.


In [ ]:
import os, pathlib, subprocess, sys
REPO = os.environ.get('TARJEEH_REPO', '')  # e.g. https://github.com/<you>/tarjeeh-ugc-ai
WORK = pathlib.Path('/kaggle/working')
PROJ = WORK/'tarjeeh-ugc-ai'
if REPO and not PROJ.exists():
    subprocess.run(['git','clone','-q',REPO,str(PROJ)], check=True)
elif not REPO:
    PROJ.mkdir(exist_ok=True)
    print('TARJEEH_REPO not set - upload backend/ and scripts/ as a Kaggle dataset instead')
sys.path.insert(0, str(PROJ))
print('project root:', PROJ)


## 4 · Job configuration

Upload the product image as a Kaggle dataset, then point `product_image` at it.
The job shape matches `examples/job.example.json`.


In [ ]:
import json, pathlib
job = {
    'product_name': 'REPLACE ME',
    'product_image': '/kaggle/input/<your-dataset>/product.jpg',
    'country': 'UAE',
    'audience': '',
    'creator': {'gender':'female','age_range':'25-35',
                'appearance_direction':'','location':'modern Dubai apartment'},
    'language': 'English', 'platform': 'Meta', 'duration': 20,
    'aspect_ratio': '9:16', 'style': 'natural handheld iPhone UGC',
    'offer': '', 'cta': 'Order Now', 'variations': 3,
    'supplied_facts': {},   # every factual claim must come from here
}
job_id = job['product_name'].lower().replace(' ','-') + '-kaggle'
project = pathlib.Path('/kaggle/working/projects')/job_id
for sub in ('product','creator','scripts','storyboards','prompts','audio','scenes','captions','renders','qa'):
    (project/sub).mkdir(parents=True, exist_ok=True)
(project/'job.json').write_text(json.dumps(job, indent=2))
print('project:', project)


## 5 · Scene prompts

Paste the prompts the `tarjeeh-ugc` skill produced. One entry per shot, ~3–5s each —
a 20s ad is composed from several shots, never one diffusion pass.


In [ ]:
scenes = [
  {'scene_index':0,'prompt':'<scene 1 prompt>','seed':1001,'num_frames':97,'speaking':True},
  {'scene_index':1,'prompt':'<scene 2 prompt>','seed':1002,'num_frames':97,'speaking':False},
  {'scene_index':2,'prompt':'<scene 3 prompt>','seed':1003,'num_frames':97,'speaking':True},
  {'scene_index':3,'prompt':'<scene 4 prompt>','seed':1004,'num_frames':97,'speaking':False},
  {'scene_index':4,'prompt':'<scene 5 prompt>','seed':1005,'num_frames':97,'speaking':True},
]
(project/'prompts'/'scenes.json').write_text(json.dumps(scenes, indent=2))
print(len(scenes), 'scenes ~', sum(s['num_frames'] for s in scenes)/24, 'seconds')


## 6 · Load Wan 2.2 TI2V-5B


In [ ]:
import torch
from diffusers import AutoencoderKLWan, WanImageToVideoPipeline
MODEL_ID = 'Wan-AI/Wan2.2-TI2V-5B-Diffusers'
# fp32 VAE with a bf16 transformer, per the model card - the Wan VAE is fragile in fp16.
vae = AutoencoderKLWan.from_pretrained(MODEL_ID, subfolder='vae', torch_dtype=torch.float32)
pipe = WanImageToVideoPipeline.from_pretrained(MODEL_ID, vae=vae, torch_dtype=torch.bfloat16)
pipe.enable_model_cpu_offload()   # T4 has 16GB; offloading keeps headroom for the VAE decode
for opt in ('enable_tiling','enable_slicing'):
    fn = getattr(pipe.vae, opt, None)
    if callable(fn): fn()
print('loaded')


## 7 · Generate scenes

Image-to-video from the real product photo — the only way the actual label survives.


In [ ]:
from diffusers.utils import export_to_video, load_image
NEG = ('cgi, plastic skin, warped label, distorted text, extra fingers, watermark, '
       'gimbal glide, studio lighting, low quality, blurry')
W, H = 720, 1280
ref = load_image(job['product_image']).resize((W, H))
for s in scenes:
    out = project/'scenes'/f"scene_{s['scene_index']:02d}.mp4"
    if out.exists():
        print('skip (already rendered)', out.name); continue
    frames = pipe(image=ref, prompt=s['prompt'], negative_prompt=NEG,
                  height=H, width=W, num_frames=s['num_frames'],
                  num_inference_steps=30, guidance_scale=5.0,
                  generator=torch.Generator('cpu').manual_seed(s['seed'])).frames[0]
    export_to_video(frames, str(out), fps=24)
    print('wrote', out.name)


## 8 · Voice — Kokoro (local, free)

A real creator recording is better for UGC. Upload one as a dataset and skip this cell.


In [ ]:
import numpy as np, soundfile as sf
from kokoro import KPipeline
SCRIPT = '<the full spoken script>'
assert '[SLOT:' not in SCRIPT, 'unresolved factual placeholder - fill it before voicing'
chunks = [a for _,_,a in KPipeline(lang_code='a')(SCRIPT, voice='af_heart', speed=1.0)]
sf.write(str(project/'audio'/'voice.wav'), np.concatenate(chunks), 24000)
print('voice written')


## 9 · Optional lip-sync — MuseTalk

Skip this if the free session is running short. A voice-over cut (B-roll, reaction shots,
product close-ups) is a perfectly good ad and never justifies a paid service.


In [ ]:
import subprocess, pathlib
RUN_LIPSYNC = False
if RUN_LIPSYNC:
    home = pathlib.Path('/kaggle/working/MuseTalk')
    if not home.exists():
        subprocess.run(['git','clone','-q','https://github.com/TMElyralab/MuseTalk',str(home)], check=True)
    print('follow MuseTalk/download_weights.sh, then run its scripts.inference')
else:
    print('skipped - shipping a voice-over cut (B-roll, reaction shots, product close-ups)')


## 10 · Captions, assembly, QA


In [ ]:
import sys, subprocess
sys.path.insert(0, str(PROJ/'scripts'))
subprocess.run([sys.executable, str(PROJ/'scripts'/'assemble_ad.py'),
                '--project', str(project), '--variation', '1', '--cta', job['cta']], check=False)
subprocess.run([sys.executable, str(PROJ/'scripts'/'qa_video.py'),
                '--project', str(project), '--variation', '1',
                '--expected-duration', str(job['duration'])], check=False)


## 11 · Download


In [ ]:
from IPython.display import FileLink, display
import pathlib
for f in sorted((project/'renders').glob('final_v*.mp4')):
    print(f, f'{f.stat().st_size/1e6:.1f} MB'); display(FileLink(str(f)))
qa = project/'qa'/'qa_report.json'
print(qa.read_text() if qa.exists() else 'no QA report')
